$$
\newcommand{\bolde}{\boldsymbol{e}}
\newcommand{\boldh}{\boldsymbol{h}}
\newcommand{\boldp}{\boldsymbol{p}}
\newcommand{\boldr}{\boldsymbol{r}}
\newcommand{\boldt}{\boldsymbol{t}}
\newcommand{\boldq}{\boldsymbol{q}}
\newcommand{\boldM}{\boldsymbol{M}}
\newcommand{\boldP}{\boldsymbol{P}}
\newcommand{\boldR}{\boldsymbol{R}}
\newcommand{\boldT}{\boldsymbol{T}}
\newcommand{\boldQ}{\boldsymbol{Q}}
$$

# High Definition (HD) Maps

<img src='./images/hd_maps_01.jpg' width='100%'>

<img src='./images/hd_maps_02.jpg' width='100%'>

In [ ]:
# Специальные import-ы
# Чтобы всё работало, выполните пункты в разделе "Настройка окружения"
import plotly
import open3d
import pykitti
import pyproj
import cv2

from IPython.display import (
    display,
    SVG,
    HTML,
    Image,
    Video,
    YouTubeVideo,
)

import typing as T
import os
import copy
import tqdm
import itertools
from collections import namedtuple
import numpy as np
import numpy.typing as npt
import pandas as pd

from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#display(HTML("<style>.container { width:100% !important; }</style>"))

<a id='_toc'></a>
# Содержание
* [Настройка окружения](#env)
* [Лидары](#lidars)
    * [Примеры лидаров](#lidars_examples)
    * [Принцип работы](#lidars_operating_principles)
    * [Данные лидара](#lidars_data)
    * [Источники ошибок](#lidars_measurement_errors)
* [Лидарные облака](#lidars_clouds)
    * [Визуализация облаков](#lidars_clouds_visualization)
    * [Операции над облаками](#lidars_clouds_operations)
        * [Линейные преобразования](#linear_transformations)
        * [Фильтрация](#pcl_filtration)
        * [Оценка признаков](#pcl_features_estimation)
        * [Геометрическая сегментация](#geometrical_segmentation)
* [Оценка позы](#pose_estimation)
    * [Point set registration](#point_set_registration)
    * [Iterative Closest Point (ICP)](#icp)
* [KITTI](#kitti)
    * [Загрузка сцен](#kitti_loading)
    * [Содержимое сцен](#kitti_content)
        * [Изображения](#kitti_images)
        * [Облака](#kitti_clouds)
        * [Локализация](#kitti_localization)
        * [Калибровки](#kitti_calibrations)
* [Источники](#sources)

<a id='env'></a>
# Настройка окружения<sup>[toc](#_toc)</sup>

Для выполнения кода из данного ноутбука лучше создать отдельное питоновское окружение. Проблема в том, что далее надо будет поставить довольно много библиотек, и, к сожалению, возможна ситуация возникнования конфликтов между ними.

Существует два способа настроить питоновское окружение для запуска кода из данного ноутбука:
1. С помощью `conda`
2. С помощью `virtualenv`

Полную информацию по управлению окружениями `conda` можно найти на [странице документации](
https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html). Далее будет просто пошаговая инструкция конкретно для наших целей.

Мы воспользуемся первым, как более простым. Предполагается, что `conda` у вас уже установлена. Для создания нового окружения с именем `sdc` и версией питона 3.12 выполняем команду:
```shell
conda create -n sdc python=3.12
```
> <span style="color:blue">Замечание</span>. Используем версию питона 3.12, потому что на текущий момент это максимальная версия питона, которую поддерживает библиотека open3d <https://www.open3d.org/docs/release/getting_started.html#python>

Если все прошло успешно, то `sdc` появится в списке окружений:
```shell
conda env list
```

Теперь переключаемся в консоли на окружение `sdc`:
```shell
conda activate sdc
```
В результате в командной строке будем видеть нечто такое:
```
(sdc) ~$
```

Теперь, когда мы успешно переключились на окружение `sdc`, начинаем устанавливаеть нужные библиотеки.

1. На всякий случай обновляем версии основных библиотек:
    ```shell
    pip install numpy --upgrade
    pip install scipy --upgrade
    pip install matplotlib --upgrade
    ```
2. Устанавливаем `jupyter lab` и `jupyter notebook` (`ipython-notebook`):
    ```shell
    pip install jupyterlab
    pip install notebook
    ```
3. Теперь самое &laquo;интересное&raquo; &mdash; ставим библиотеки, которые нужны конкретно для данного ноутбука:
    ```shell
    pip install pydantic
    pip install opencv-python
    pip install plotly
    pip install open3d
    pip install pykitti
    pip install pyproj
    ```

Если все прошло успешно, то закрываем данный ноутбук, в консоли переключаемся на окружение `sdc`, и находясь в данном окружение запускаемся заново
```
(sdc) ...$ ls
hd_maps.ipynb ...
(sdc) ...$ jupyter notebook
```
После перезапуска все импорты ниже должны работать.

> <span style="color:red">Замечание 1</span>. Если вдруг у вас не не будут нормально рендериться latex-вставки, то попробуйте вручную поставить `MathJax` и перезапустить `jupyter notebook`:
    ```shell
    pip install jupyterlab-mathjax3
    ```

> <span style="color:red">Замечание 2</span>. Если будут какие-то проблемы с 3D-визуализацией `plotly`, то стоит попробовать запустиь ноутбук с командой `--no-browser`
> ```shell
> (sdc) ...$ jupyter notebook --no-browser
> ```
> и уже затем открыть ссылку вида `http://127.0.0.1:8888/tree?token=...`

<a id='lidars'></a>
# Лидары<sup>[toc](#_toc)</sup>

> **LiDAR &mdash; Light Detection and Ranging**

* [Примеры лидаров](#lidars_manufacturers)

| Rank | Company (Ticker) | Market Cap (USD) | Data Source \& Notes |
| :--- | :--- | :--- | :--- |
| **#1** | **Hesai** (HSAI.US) | **\~\\$3.70 Billion** | Data from December 2025. |
| **#2** | **Ouster** (OUST) | **\~\\$2.09 Billion** | Based on a 52-week high report from October 2025. However, a more recent figure from Morningstar (February 2026) shows a market cap of **\$1.12 Billion**, suggesting significant volatility. Ouster merged with Velodyne in 2023. |
| **#3** | **RoboSense** (2498.HK) | **\~\$2.09 Billion** | Calculated from HKD 16.24 billion reported in February 2026 (value for late 2025). |
| **#4** | **Innoviz Technologies** (INVZ) | **\~\\$198.9-\\$221.1 Million** | Data points vary slightly. Stock Analysis reports \\$198.87M as of February 2, 2026, while Public.com shows \\$221.06M as of January 20, 2026. |
| **#5** | **MicroVision** (MVIS) | **\~\$249 Million** | Based on data from MarketScreener (November 2025), which lists the market cap at \$286M USD. |
| **#6** | **AEye** (LIDR) | **\~\$115.4 Million** | Enterprise value data from December 2025 breaks down a market cap of \\$115.447M . |
| **#7** | **Luminar Technologies** (LAZR) | **\~\$47.1 Million** | Based on a market cap of \\$47.058M as of December 19, 2025. It is worth noting that another source from early December 2025 lists a much higher value of \\$1.24B, but this is not corroborated by the detailed historical chart. |
| **#8** | **Cepton** (CPTN) | **Acquired / N/A** | Cepton was delisted after being acquired by Koito. Its last reported market cap before being acquired was **\$52.17 million** in early January 2025. |

<a id='lidars_examples'></a>
## Примеры лидаров<sup>[toc](#_toc)</sup>
* [Hesai](#hesai)
* [Velodyne](velodyne)
* [RoboSense](#robosense)
* [Waymo](waymo)
* [Yandex](yandex)

<a id='hesai'></a>
### Hesai<sup>[toc](#_toc)</sup>

In [ ]:
Image('./images/Hesai_ETX.png', width='100%')

In [ ]:
Image('./images/Hesai_QT128.png', width='100%')

<a id='velodyne'></a>
### Velodyne<sup>[toc](#_toc)</sup>

In [ ]:
Image('./images/velodyne_lidar_00.png', width='100%')

In [ ]:
Image('./images/velodyne_lidar_01.png', width='100%')

In [ ]:
Image('./images/velodyne_lidar_cloud_HDL-64E.jpg', width='100%')

In [ ]:
YouTubeVideo('tZ8WbSNsNaU', width='100%', height=500)

<a id='robosense'></a>
### RoboSense<sup>[toc](#_toc)</sup>

In [ ]:
Image('./images/RoboSense_RubyPlus.jpg', width='100%')

In [ ]:
Image('./images/RoboSense_RubyPlus_2.png', width='100%')

In [ ]:
Video(filename='./videos/RoboSense_RubyPlus_1.mp4', width=1024)

In [ ]:
Video(filename='./videos/RoboSense_RubyPlus_2.mp4', width=1024)

<a id='waymo'></a>
### Waymo<sup>[toc](#_toc)</sup>

In [ ]:
Image('./images/waymo_lidar_cloud_00.png', width='100%')

<a id='yandex_lidars'></a>
### Yandex<sup>[toc](#_toc)</sup>

#### Модели<sup>[toc](#_toc)</sup>

In [ ]:
Image('./images/yandex_lidar_00.jpeg')

In [ ]:
Image('./images/yandex_lidar_01.jpeg')

In [ ]:
Image('./images/yandex_lidar_02.jpeg')

<a id='yandex_lidar_clouds'></a>
### Облака лидара<sup>[toc](#_toc)</sup>

**Так выглядит изображение с лидара. Можно увидеть дороги, пешеходов, припаркованные машины, стоянку самокатов и другие объекты вокруг беспилотного автомобиля**

In [ ]:
Image('./images/yandex_lidar_cloud_00.png', width='100%')

In [ ]:
Image('./images/yandex_lidar_cloud_01.jpeg', width='100%')

In [ ]:
Image('./images/yandex_lidar_cloud_02.jpeg', width='100%')

In [ ]:
Image('./videos/yandex_lidar_02.gif', width='100%')

In [ ]:
Video('./videos/Yandex Software-Defined LiDAR.mp4', width=970)

In [ ]:
YouTubeVideo('lPclzuDtmTs', width='100%', height=500)

<a id='lidars_operating_principles'></a>
## Принцип работы<sup>[toc](#_toc)</sup>
* [Time-of-flight ranging](#time_of_flight_ranging)
* [Интенсивность](#intensity)
* [Сканирование пространства](#scanning)

<a id='time_of_flight_ranging'></a>
### Time-of-flight ranging<sup>[toc](#_toc)</sup>

Лидар работает по принципу [**time-of-flight ranging**-а](https://en.wikipedia.org/wiki/Time-of-flight_camera). По такому же принципу работают радары и сонары.

Терминология:
* Time-of-Flight
* Time-of-Arrival
* Round trip time (RTT)
* Round trip distance (RTD)

In [ ]:
display(SVG('https://upload.wikimedia.org/wikipedia/commons/f/f1/20200501_Time_of_flight.svg'))

Пусть
* $t$ &mdash; время от отправки импулься до его получения, т.е. RTT (Round Trip Time)
* $c$ &mdash; скорость света
Тогда расстояние $r$ до мишени/объекта наблюдения можно __оценить__ следующим образом:
$$
r \approx \frac{c \Delta t}{2}
$$

<a id='intensity'></a>
### Интенсивность<sup>[toc](#_toc)</sup>

![](https://res.mdpi.com/sensors/sensors-15-28099/article_deploy/html/images/sensors-15-28099-g001-1024.png)

<a id='scanning'></a>
### Сканирование пространства<sup>[toc](#_toc)</sup>

Вращение зеркала-отражателя вокруг вертикальной оси позволяет получить 2d-срез окружающего пространства. Чтобы получить паттерн 3d-сканирования добавляют вращение зеркальца вокруг горизонтальной оси.
![](videos/general_rotating_lidar.gif)

Лучей может быть много:
![](https://eckop.com/wp-content/uploads/2018/07/LIDAR_02-1024x366.png)

Паттерн сканирования велодайна:

![](./images/velodyne_lidar_structure_00.png)

![](./images/velodyne_lidar_structure_01.png)

[Static Calibration and Analysis of the Velodyne HDL-64E S2 for High Accuracy Mobile Scanning](https://www.mdpi.com/2072-4292/2/6/1610)

<a id='lidars_data'></a>
## Данные с лидара<sup>[toc](#_toc)</sup>

### "Сырые" показания лидара<sup>[toc](#_toc)</sup>
* Горизонатльный поворот зеркальца (galvo horizontal encoder value)
* Вертикальный поворот зеркальца (galvo vertical encoder value)
* время полета (time-of-flight)

### Сферическая система координат<sup>[toc](#_toc)</sup>

Положение точки $P$ в сферической системе координат определяется тройкой $(r, \theta, \varphi)$, где
* $r \ge 0$ &mdash; **расстояние** (**range/distance**)
* $\theta \in [-\pi/2, \pi/2]$ &mdash; **зенитный** или **полярный** угол (**elevation angle**)
* $\varphi \in [0, 2\pi)$ &mdash; **азимутальный** угол (**azimuth angle**)

Иногда встречаются альтернативные обозначения:
* $r$ &mdash; **r**ange
* $\alpha$ &mdash; **a**zimuth angle
* $\varepsilon$ &mdash; **e**levation angle

In [ ]:
display(SVG('./images/spherical_coordinate_system.svg'))

### Декартова система координат<sup>[toc](#_toc)</sup>

\begin{align*}
\begin{pmatrix}
x\\
y\\
z
\end{pmatrix} =
\begin{pmatrix}
r \cos\theta \cos\phi \\ 
r \cos\theta \sin\phi\\
r \sin\theta
\end{pmatrix}
\end{align*}

\begin{align*}
\begin{pmatrix}
r\\
\phi\\
\theta\\
\end{pmatrix} = \boldh(x, y, z) = \begin{pmatrix}
\sqrt{x^2 + y^2 + z^2}\\
\arctan \left(\frac{y}{x}\right)\\
\arcsin \left(\frac{z}{\sqrt{x^2 + y^2 + z^2}}\right)
\end{pmatrix}
\end{align*}

<a id='lidars_measurement_errors'></a>
## Источники ошибок<sup>[toc](#_toc)</sup>
* Неточность в определении времени прибытия сигнала (из-за peak detector-а)
* Неточности в определении внутренних механических частей лидара (зеркальца и т.п.)
* Взаимодействие со средой
* Взаимодействие с поверхностью отражения

Пусть $\boldp = (x, y, z)^T  \in \mathbb{R}^3$ &mdash; реальная точка, от которой отразился сигнал, тогда в первом приближении можно моделировать ошибки наблюдения нормальным шумом:
$$
\begin{pmatrix}
\hat{r}\\
\hat{\phi}\\
\hat{\theta}\\
\end{pmatrix} = \hat{\boldh}(x, y, z) = \boldh(x, y, z) + \boldq,
$$
где $\boldq \sim \mathcal{N}(\boldsymbol{0}, Q)$.

Ещё одна причина искажений в наблюдениях &mdash; движение устройства, на котором установлен лидар (робота, машины). В таком случае могут возникать двоения, растяжени и сжатия объектов (впрочем, как и много всего прочего).

<a id='lidar_intrinsics_calibration'></sup>
## Калибровка внутренних параметров лидара<sup>[toc](#_toc)</sup>
* Внутренние параметры лидара (lidar intrinsics)
* Калибровка внутренних параметров лидара (lidar intrinsics calibration)

In [ ]:
import abc
from pydantic import BaseModel
from collections import namedtuple

Point = namedtuple('Point', ['x', 'y'])
Vector = namedtuple('Vector', ['x', 'y'])
LineSegment = namedtuple('LineSegment', ['first', 'second'])
Ray = namedtuple('Ray', ['origin', 'direction'])
Pose = namedtuple('Pose', ['x', 'y', 'yaw'])

PointXY = namedtuple('PointXY', ['x', 'y'])
PointDA = namedtuple('PointDA', ['distance', 'azimuth'])

def convert_point_da_to_xy(point_da: PointDA) -> PointXY:
    assert isinstance(point_da, PointDA)
    distance, azimuth = point_da.distance, point_da.azimuth
    x = distance * np.cos(azimuth)
    y = distance * np.sin(azimuth)
    return PointXY(x, y)

def convert_point_xy_to_da(point_xy, PointXY) -> PointDA:
    assert isinstance(point_xy, PointXY)
    azimuth = np.arctan2(point_xy.y, point_xy.x)
    distance = np.hypot(point_xy.x, point_xy.y)
    return PointDA(distance, azimuth)


def convert_numpy_to_point(point: npt.ArrayLike) -> Point:
    point = np.array(point, dtype=np.float64, copy=False)
    assert point.shape == (2,)
    return Point(*point)

def convert_point_to_numpy(point: Point) -> np.ndarray:
    assert isinstance(point, Point)
    return np.array([point.x, point.y], dtype=np.float64)


def convert_numpy_to_ray(ray: npt.ArrayLike) -> Ray:
    ray = np.array(ray, dtype=np.float64, copy=False)
    assert ray.shape == (4,)
    ray_origin = Point(x=ray[0], y=ray[1])
    ray_direction = Vector(x=ray[2], y=ray[3])
    return Ray(origin=ray_origin, direction=ray_direction)


class LidarModelBase(abc.ABC):
    @abc.abstractmethod
    def get_rays(self) -> np.ndarray:
        ...


class OmniDirectionalLidarModel(LidarModelBase):
    class Parameters(BaseModel):
        num_rays: int
        angles_corrections: T.List[float]
    
    class RawPoint(BaseModel):
        distance: float
        laser_id: int

    def __init__(self, params):
        assert isinstance(params, OmniDirectionalLidarModel.Parameters)
        assert params.num_rays == len(params.angles_corrections)
        self._params = copy.deepcopy(params)

    @property
    def params(self):
        return self._params
    
    def get_lasers_angles(self) -> np.ndarray:
        angles = np.linspace(0, 2 * np.pi, self._params.num_rays)
        return angles + self._params.angles_corrections
        
    def get_rays(self) -> np.ndarray:
        angles = self.get_lasers_angles()
        rays_directions = np.hstack([np.cos(angles)[:, None], np.sin(angles)[:, None]])
        assert rays_directions.shape == (self._params.num_rays, 2) and rays_directions.dtype == np.float64
        rays_origins = np.zeros(shape=(self._params.num_rays, 2), dtype=np.float64)
        return np.hstack([rays_origins, rays_directions])

    def restore_numpy_cloud_xy(self, raw_points) -> np.ndarray:
        lasers_angles = self.get_lasers_angles()
        numpy_cloud_xy = []
        for raw_point in raw_points:
            assert isinstance(raw_point, OmniDirectionalLidarModel.RawPoint)
            laser_angle = lasers_angles[raw_point.laser_id]
            x = raw_point.distance * np.cos(laser_angle)
            y = raw_point.distance * np.sin(laser_angle)
            numpy_cloud_xy.append((x, y))
        return np.array(numpy_cloud_xy, dtype=np.float64)

#### Лучи лидара в собственной системе координат<sup>[toc](#_toc)</sup>

In [ ]:
num_rays = 32
lidar_model_params = OmniDirectionalLidarModel.Parameters(
    num_rays=num_rays,
    angles_corrections=np.zeros(num_rays),
)
lidar_model = OmniDirectionalLidarModel(lidar_model_params)


def draw_lidar(ax, lidar_model: LidarModelBase):
    lidar_rays = lidar_model.get_rays()
    for lidar_ray in lidar_rays:
        plt.arrow(*lidar_ray, head_width=0.05, zorder=4)
    ax.arrow(0, 0, 1.5, 0, color='r', label='OX_lidar', head_width=0.1, zorder=3)
    ax.arrow(0, 0, 0, 1.5, color='g', label='OY_lidar', head_width=0.1, zorder=3)


ax = plt.figure(figsize=(6, 6)).gca()
ax.set_xlim([-2, 2])
ax.set_ylim([-2, 2])
ax.set_aspect('equal')
draw_lidar(ax, lidar_model)
ax.grid(which='both', linestyle='--')
ax.legend();

del draw_lidar

#### Лучи лидара во внешней системе координат<sup>[toc](#_toc)</sup>

In [ ]:
def get_lidar_rays(lidar_model: LidarModelBase, lidar_pose: T.Optional[Pose] = None) -> np.ndarray:
    assert isinstance(lidar_model, LidarModelBase)

    rays_in_lidar = lidar_model.get_rays()
    assert rays_in_lidar.shape == (lidar_model.params.num_rays, 4) and rays_in_lidar.dtype == np.float64

    if lidar_pose is None:
        return rays_in_lidar

    lidar_to_world_rotation_matrix = np.array([
        [np.cos(lidar_pose.yaw), -np.sin(lidar_pose.yaw)],
        [np.sin(lidar_pose.yaw), np.cos(lidar_pose.yaw)],
    ], dtype=np.float64)
    lidar_to_world_translation_vector = np.array([lidar_pose.x, lidar_pose.y], dtype=np.float64)

    rays_origins_in_lidar = rays_in_lidar[:, :2]
    rays_directions_in_lidar = rays_in_lidar[:, 2:]

    rays_origins_in_world = rays_origins_in_lidar + lidar_to_world_translation_vector[None, :]
    rays_directions_in_world = np.dot(rays_directions_in_lidar, lidar_to_world_rotation_matrix.T)

    return np.hstack([rays_origins_in_world, rays_directions_in_world])

In [ ]:
lidar_pose = Pose(x=3, y=1, yaw=np.deg2rad(17))


def draw_lidar(ax, lidar_model: LidarModelBase, lidar_pose: T.Optional[Pose] = None):
    lidar_rays = get_lidar_rays(lidar_model, lidar_pose)
    for lidar_ray in lidar_rays:
        plt.arrow(*lidar_ray, head_width=0.05, zorder=4)

    lidar_ox_axis = np.array([1.5, 0.0])
    lidar_oy_axis = np.array([0.0, 1.5])
    if lidar_pose is not None:
        lidar_to_world_rotation_matrix = np.array([
            [np.cos(lidar_pose.yaw), -np.sin(lidar_pose.yaw)],
            [np.sin(lidar_pose.yaw), np.cos(lidar_pose.yaw)],
        ], dtype=np.float64)
        lidar_ox_axis = lidar_to_world_rotation_matrix @ lidar_ox_axis
        lidar_oy_axis = lidar_to_world_rotation_matrix @ lidar_oy_axis
        
    ax.arrow(lidar_pose.x, lidar_pose.y, lidar_ox_axis[0], lidar_ox_axis[1], color='r', label='OX_lidar', head_width=0.1, zorder=3)
    ax.arrow(lidar_pose.x, lidar_pose.y, lidar_oy_axis[0], lidar_oy_axis[1], color='g', label='OY_lidar', head_width=0.1, zorder=3)


ax = plt.figure(figsize=(6, 6)).gca()
ax.set_xlim([1, 5])
ax.set_ylim([-1, 3])
ax.set_aspect('equal')
draw_lidar(ax, lidar_model, lidar_pose)
ax.grid(which='both', linestyle='--')
ax.legend();

del lidar_pose

#### Генерирование лидарного облака<sup>[toc](#_toc)</sup>

<span style="color:blue">**Найдем уравнение прямой, которая проходит через две точки $\boldsymbol{a}$ и $\boldsymbol{b}$.**</span>

Несколько видов уравнения прямой:
1. $Ax + By + C = 0$
2. $\langle\boldsymbol{n}, \boldsymbol{p}\rangle = d$
3. $\boldsymbol{p_0} + t \boldsymbol{d}$

Остановимся на первом типе:
$$
\begin{cases}
A a_x + B a_y + C = 0\\
A b_x + B b_y + C = 0
\end{cases}
\Rightarrow
\begin{cases}
A = -b_y + a_y\\
B = b_x - a_x\\
C = a_x b_y - a_y b_x\\
\end{cases}
$$

$$
\begin{cases}
\boldsymbol{n} = [-B, A]^T\\
d = -C
\end{cases}
$$

<span style="color:blue">**Найдем координату точки пересечения двух прямых.**</span>

Даны две прямые $A_1 x + B_1 y + C_1 = 0$ и $A_2 x + B_2 y + C_2 = 0$. Требуется найти их точку пересечения $(x, y)^T$:
$$
\begin{cases}
A_1 x + B_1 y + C_1 = 0\\
A_2 x + B_2 y + C_2 = 0
\end{cases}
$$

In [ ]:
def get_line_coefficients_from_two_points(
        a: T.Union[Point, npt.ArrayLike],
        b: T.Union[Point, npt.ArrayLike]) -> np.ndarray:
    if isinstance(a, Point):
        a = (a.x, a.y)
    if isinstance(b, Point):
        b = (b.x, b.y)
    a = np.array(a, dtype=np.float64, copy=False)
    b = np.array(b, dtype=np.float64, copy=False)
    A = -b[1] + a[1]
    B = b[0] - a[0]
    C = a[0] * b[1] - a[1] * b[0]
    return np.array((A, B, C), dtype=np.float64)


def intersect_ray_with_line_segment(ray: Ray, line_segment: LineSegment) -> T.Optional[Point]:
    ray_direction = np.array([ray.direction.x, ray.direction.y], dtype=np.float64)
    ray_start = np.array([ray.origin.x, ray.origin.y], dtype=np.float64)
    ray_end = np.array([ray.origin.x + ray.direction.x, ray.origin.y + ray.direction.y], dtype=np.float64)

    line_start = np.array([line_segment.first.x, line_segment.first.y], dtype=np.float64)
    line_end = np.array([line_segment.second.x, line_segment.second.y], dtype=np.float64)
    assert np.any(line_start != line_end)

    A1, B1, C1 = get_line_coefficients_from_two_points(ray_start, ray_end)
    A2, B2, C2 = get_line_coefficients_from_two_points(line_start, line_end)

    a = np.array([
        [A1, B1],
        [A2, B2],
    ])
    b = np.array([-C1, -C2])
    if np.linalg.cond(a) > 1e6:
        # Parallel lines
        return None

    x = np.linalg.solve(a, b)

    if np.dot(x - ray_start, ray_direction) < 0:
        # Точка пересечения в противположном направлении от луча
        return None

    # Определяем, лежит ли точка пересечения в пределах отрезка прямой
    ray_start_to_line_start_3d = np.append(line_start - ray_start, 0)
    ray_start_to_line_end_3d = np.append(line_end - ray_start, 0)
    ray_start_to_x_3d = np.append(x - ray_start, 0)
    c1 = np.cross(ray_start_to_x_3d, ray_start_to_line_start_3d)
    c2 = np.cross(ray_start_to_x_3d, ray_start_to_line_end_3d)
    if np.sign(c1[2]) == np.sign(c2[2]):
        # Точка пересечения лежит за пределами грани отрезка
        return None

    return Point(x[0], x[1])

In [ ]:
ray = Ray(Point(1, 1), Vector(1, 1))

# Точка пересечения есть
line_segment = LineSegment(Point(0, 2), Point(3, 2))
print(intersect_ray_with_line_segment(ray, line_segment))

# Параллельные прямые
line_segment = LineSegment(Point(1, 2), Point(2, 3))
print(intersect_ray_with_line_segment(ray, line_segment))

In [ ]:
class Room:
    def __init__(self, contour: T.List[T.Tuple[float, float]], label: T.Optional[str] = None):
        assert len(contour) > 1
        self._contour = copy.deepcopy(contour)
        self._label = label

    def get_x_data(self):
        return [point[0] for point in self._contour] + [self._contour[0][0]]

    def get_y_data(self):
        return [point[1] for point in self._contour] + [self._contour[0][1]]

    def get_room_sides(self) -> T.List[LineSegment]:
        line_segments = []
        for a, b in zip(self._contour[:-1], self._contour[1:]):
            line_segment = LineSegment(first=Point(*a), second=Point(*b))
            line_segments.append(line_segment)
        line_segment = LineSegment(first=Point(*self._contour[-1]), second=Point(*self._contour[0]))
        line_segments.append(line_segment)
        return line_segments
    
    def draw(self, ax):
        ax.plot(self.get_x_data(), self.get_y_data(), label=self._label)


room = Room([(-5,-5), (-4, 3), (4, 4), (4, -6)], label='room')
fig, ax = plt.subplots(figsize=(6, 6))
room.draw(ax)

In [ ]:
def generate_raw_lidar_cloud(lidar_model: LidarModelBase, lidar_pose: Pose, room: Room):
    lidar_rays = get_lidar_rays(lidar_model, lidar_pose)
    raw_lidar_cloud = []
    for laser_id, lidar_ray in enumerate(lidar_rays):
        lidar_ray = convert_numpy_to_ray(lidar_ray)
        lidar_points = []
        for room_side in room.get_room_sides():
            lidar_point = intersect_ray_with_line_segment(lidar_ray, room_side)
            if lidar_point is not None:
                lidar_points.append(convert_point_to_numpy(lidar_point))
        assert len(lidar_points) > 0
        lidar_points = np.array(lidar_points)
        lidar_distances = np.linalg.norm(lidar_points, axis=1)
        
        min_distance = lidar_distances[np.argmin(lidar_distances)]
        raw_lidar_cloud.append(lidar_model.RawPoint(distance=min_distance, laser_id=laser_id))

    return raw_lidar_cloud

In [ ]:
lidar_position_x = 0  # np.mean(room.get_x_data()[:-1])
lidar_position_y = 0  # np.mean(room.get_y_data()[:-1])
lidar_orientation_yaw = np.deg2rad(0)
lidar_pose = Pose(lidar_position_x, lidar_position_y, lidar_orientation_yaw)

raw_lidar_cloud = generate_raw_lidar_cloud(lidar_model, lidar_pose, room)
restored_lidar_cloud_xy = lidar_model.restore_numpy_cloud_xy(raw_lidar_cloud)

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect('equal')
room.draw(ax)
ax.plot(lidar_position_x, lidar_position_y, marker='*', color='r', label='Lidar origin')
ax.scatter(restored_lidar_cloud_xy[:, 0], restored_lidar_cloud_xy[:, 1], color='r', zorder=2, label='Lidar cloud')
draw_lidar(ax, lidar_model, lidar_pose)
ax.grid(which='both', linestyle='--', color='k', alpha=0.5);
ax.legend();

del raw_lidar_cloud, restored_lidar_cloud_xy

**Теперь создадим модель лидара с неравномерным расположением лучей**

In [ ]:
num_rays = 32
random_state = np.random.RandomState(1234456)
angle_step = 2 * np.pi / num_rays
gt_angles_corrections = random_state.uniform(-0.25 * angle_step, 0.25 * angle_step, size=num_rays)

real_lidar_model_params = OmniDirectionalLidarModel.Parameters(
    num_rays=num_rays,
    angles_corrections=gt_angles_corrections,
)
real_lidar_model = OmniDirectionalLidarModel(real_lidar_model_params)

ax = plt.figure(figsize=(6, 6)).gca()
ax.set_xlim([-2, 2])
ax.set_ylim([-2, 2])
ax.set_aspect('equal')
draw_lidar(ax, real_lidar_model, lidar_pose)
ax.grid(which='both', linestyle='--')
ax.legend();

In [ ]:
raw_lidar_cloud = generate_raw_lidar_cloud(real_lidar_model, lidar_pose, room)

restored_lidar_cloud_xy = lidar_model.restore_numpy_cloud_xy(raw_lidar_cloud)

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect('equal')
room.draw(ax)
ax.plot(lidar_position_x, lidar_position_y, marker='*', color='r', label='Lidar origin')
ax.scatter(restored_lidar_cloud_xy[:, 0], restored_lidar_cloud_xy[:, 1], color='r', zorder=2, label='Lidar cloud')
draw_lidar(ax, lidar_model, lidar_pose)
ax.grid(which='both', linestyle='--', color='k', alpha=0.5);
ax.legend();

del raw_lidar_cloud, restored_lidar_cloud_xy

<a id='lidars_clouds'></a>
# Лидарные облака<sup>[toc](#_toc)</sup>
* [Визуализация облаков](#lidar_clouds_visualization)
* [Операции над облаками](#lidars_clouds_operations)

<a id='lidars_clouds_visualization'></a>
## Визуализация облаков<sup>[toc](#_toc)</sup>

In [ ]:
import plotly.offline as py
py.init_notebook_mode(connected=True)

from sdc.pcl.tools.plotly_visualization import (
    create_plotly_figure,
    plot_cloud,
    apply_min_max_scaling,
    convert_values_to_rgba_tuples_f64,
)

#### Случайное облако<sup>[toc](#_toc)</sup>

In [ ]:
num_points = 100
cloud_xyz = np.random.normal(size=(num_points, 3))

colors = apply_min_max_scaling(cloud_xyz[:, 2])

colors = convert_values_to_rgba_tuples_f64(colors, cmap='viridis')
figure = create_plotly_figure(bgcolor='black')
plot_cloud(
    cloud_xyz,
    colors=colors,
    labels=np.linalg.norm(cloud_xyz, axis=1),
    marker_size=3, figure=figure).show()
del num_points, cloud_xyz, colors

#### Bunny<sup>[toc](#_toc)</sup>

In [ ]:
data = pd.read_csv('./datasets/pcl/bunny.txt', header=None, sep=' ', names=['x', 'y', 'z'])
cloud_xyz = data.to_numpy()
x = cloud_xyz[:, 0]
y = cloud_xyz[:, 2]
z = cloud_xyz[:, 1]
cloud_xyz = np.stack([x, y, z]).T
print(cloud_xyz.shape)
plot_cloud(cloud_xyz, marker_size=5).show()
del data, cloud_xyz

<a id='lidars_clouds_operations'></a>
## Операции над облаками<sup>[toc](#_toc)</sup>

In [ ]:
from scipy.spatial.transform import Rotation
from sdc.pcl.common.cloud_io import read_point_cloud_o3d, read_point_cloud_xyz
from sdc.pcl.common.convert_point_cloud import convert_point_cloud_o3d_to_xyz
from sdc.pcl.common.transform_point_cloud import transform_point_cloud_xyz
from sdc.pcl.filters.voxel_grid import apply_voxel_grid

<a id='linear_transformations'></a>
### Линейные преобразования<sup>[toc](#_toc)</sup>

**Базовые операции**:
* Сдвиг (translation)
* Поворот (rotation)
* Масштабирование (scaling)

**Композитные операции**:
* Афинное преобразование (используется редко, так как не физично)
* Изометрическое преобразование (__преобразование системы координат__)

<a id='example_stanford_bunny'></a>
#### Пример. Stanford Bunny<sup>[toc](#_toc)</sup>

http://graphics.stanford.edu/data/3Dscanrep/

> **Задача** Загрузить данные сканирования датасета и смержить в единое облако с использованием информации о том, из каких поз производилось сканирование

Читаем облака:

In [ ]:
dataset_path = './datasets/stanford/bunny/data'
clouds_names = [
    'bun000.ply',
    'bun045.ply',
    'bun090.ply',
    'bun180.ply',
    'bun270.ply',
    'top2.ply',
    'top3.ply',
    'bun315.ply',
    'chin.ply',
    'ear_back.ply',
]

cloud_o3d_by_name = {}
for cloud_name in clouds_names:
    cloud_o3d = read_point_cloud_o3d(os.path.join(dataset_path, cloud_name))
    cloud_o3d_by_name[cloud_name] = cloud_o3d
    print(cloud_o3d)
    del cloud_o3d

Читаем позы:

In [ ]:
conf_name = 'bun.conf'

pose_by_cloud_name = {}
with open(os.path.join(dataset_path, conf_name), 'r') as f:
    for line in f.readlines():
        line = line.split()
        if len(line) == 0:
            continue
        if line[0] != 'bmesh':
            continue
        _, cloud_name, x, y, z, qx, qy, qz, qw = line
        x = float(x)
        y = float(y)
        z = float(z)
        qx = float(qx)
        qy = float(qy)
        qz = float(qz)
        qw = float(qw)
        rotation_matrix = np.linalg.inv(Rotation.from_quat([qx, qy, qz, qw]).as_matrix())
        translation_vector = np.array([x, y, z])
        transform_matrix = np.eye(4)
        transform_matrix[:3, :3] = rotation_matrix
        transform_matrix[:3, 3] = translation_vector
        pose_by_cloud_name[cloud_name] = transform_matrix

Получаем XYZ-облака:

In [ ]:
cloud_xyz_by_name = {}
transformed_cloud_xyz_by_name = {}
for cloud_name, cloud_o3d in cloud_o3d_by_name.items():
    cloud_xyz_by_name[cloud_name] = convert_point_cloud_o3d_to_xyz(cloud_o3d)
    transformed_cloud_xyz_by_name[cloud_name] =\
        transform_point_cloud_xyz(cloud_xyz_by_name[cloud_name], pose_by_cloud_name[cloud_name])

Мержим трансформированные облака в единое облако:

In [ ]:
merged_cloud_xyz = []
for cloud_name in clouds_names:
    merged_cloud_xyz.append(transformed_cloud_xyz_by_name[cloud_name])
merged_cloud_xyz = np.vstack(merged_cloud_xyz)
print(f'Merged cloud shape: {merged_cloud_xyz.shape}')

Отрисовывать облако такого размера для `plotly` &mdash; довольно сложная задача. Поэтому уменьшим его размер с помощью вокселизации:

In [ ]:
print(f'Merged cloud shape before voxel grid: {merged_cloud_xyz.shape}')
merged_cloud_xyz = apply_voxel_grid(merged_cloud_xyz, voxel_size=0.0015)
print(f'Merged cloud shape after voxel grid: {merged_cloud_xyz.shape}')

Визуализируем смерженное вокселизируемое облако:

In [ ]:
figure = create_plotly_figure()
plot_cloud(merged_cloud_xyz, figure=figure)
figure.show()

<a id='pcl_filtration'></a>
### Фильтрация<sup>[toc](#_toc)<sup>

Сэмплирование облака размера $N$:
* Случайное:
    * Fisher-Yates shuffle (выбираем случайное подмножество размера $K$
* Детерминированное
    * Voxel grid
    * Uniform sampling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib import cm
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple, Any


# =============================
# Data structures
# =============================
@dataclass
class GridInfo2D:
    bounds: Tuple[float, float, float, float]  # (x_min, x_max, y_min, y_max)
    cell_size: float
    num_cells_x: int
    num_cells_y: int


@dataclass
class OccupancyGrid2D:
    grid_info: GridInfo2D
    occupied_cells: List[Tuple[int, int]]
    cell_point_counts: np.ndarray  # (K,)
    cell_metadata: List[Dict[str, Any]]  # (K,)


# =============================
# 1) Point cloud generation
# =============================
def generate_point_cloud_2d(num_points: int = 2500, seed: int = 7) -> np.ndarray:
    rng = np.random.default_rng(seed)

    num_points_first = num_points // 2
    num_points_second = num_points - num_points_first

    cluster_1 = rng.normal(loc=(0.30, 0.30), scale=0.07, size=(num_points_first, 2))
    cluster_2 = rng.normal(loc=(0.75, 0.65), scale=0.10, size=(num_points_second, 2))
    outliers = rng.uniform(low=0.0, high=1.0, size=(max(20, num_points // 50), 2))

    points = np.vstack([cluster_1, cluster_2, outliers])
    points = np.clip(points, 0.0, 1.0)
    return points


# =============================
# 2) Grid helpers (shared)
# =============================
def compute_grid_info_2d(cell_size: float, bounds: Tuple[float, float, float, float]) -> GridInfo2D:
    x_min, x_max, y_min, y_max = bounds
    num_cells_x = int(np.ceil((x_max - x_min) / cell_size))
    num_cells_y = int(np.ceil((y_max - y_min) / cell_size))
    return GridInfo2D(bounds=bounds, cell_size=cell_size, num_cells_x=num_cells_x, num_cells_y=num_cells_y)


def compute_cell_indices_2d(points: np.ndarray, grid_info: GridInfo2D) -> Tuple[np.ndarray, np.ndarray]:
    x_min, x_max, y_min, y_max = grid_info.bounds
    cell_size = grid_info.cell_size

    x_cell_index = np.floor((points[:, 0] - x_min) / cell_size).astype(int)
    y_cell_index = np.floor((points[:, 1] - y_min) / cell_size).astype(int)

    x_cell_index = np.clip(x_cell_index, 0, grid_info.num_cells_x - 1)
    y_cell_index = np.clip(y_cell_index, 0, grid_info.num_cells_y - 1)
    return x_cell_index, y_cell_index


def compute_cell_center_2d(cell_index_x: int, cell_index_y: int, grid_info: GridInfo2D) -> np.ndarray:
    x_min, x_max, y_min, y_max = grid_info.bounds
    cell_size = grid_info.cell_size
    return np.array([x_min + (cell_index_x + 0.5) * cell_size,
                     y_min + (cell_index_y + 0.5) * cell_size])


def group_points_by_cell_2d(
    points: np.ndarray,
    grid_info: GridInfo2D,
) -> Tuple[np.ndarray, Dict[Tuple[int, int], np.ndarray]]:
    """
    Returns:
        filtered_points: points inside bounds
        cell_to_point_indices: dict(cell_index -> np.ndarray(indices into filtered_points))
    """
    x_min, x_max, y_min, y_max = grid_info.bounds

    inside_bounds_mask = (
        (points[:, 0] >= x_min) & (points[:, 0] <= x_max) &
        (points[:, 1] >= y_min) & (points[:, 1] <= y_max)
    )
    filtered_points = points[inside_bounds_mask]

    x_cell_index, y_cell_index = compute_cell_indices_2d(filtered_points, grid_info)

    cell_to_indices_list: Dict[Tuple[int, int], List[int]] = {}
    for point_index in range(len(filtered_points)):
        cell_index = (int(x_cell_index[point_index]), int(y_cell_index[point_index]))
        cell_to_indices_list.setdefault(cell_index, []).append(point_index)

    cell_to_point_indices: Dict[Tuple[int, int], np.ndarray] = {
        cell_index: np.array(index_list, dtype=int)
        for cell_index, index_list in cell_to_indices_list.items()
    }
    return filtered_points, cell_to_point_indices


# =============================
# 3) Occupancy grid computation (independent of sampling)
# =============================
def default_cell_metadata_function(
    cell_points: np.ndarray,
    cell_center: np.ndarray,
) -> Dict[str, Any]:
    """
    Placeholder for future extensions. Right now stores a couple of useful values.
    You can later add: covariance, max radius, entropy, etc.
    """
    distances = np.linalg.norm(cell_points - cell_center[None, :], axis=1)
    return {
        "cell_center": cell_center,
        "mean_distance_to_center": float(distances.mean()) if len(distances) > 0 else 0.0,
        "min_distance_to_center": float(distances.min()) if len(distances) > 0 else 0.0,
    }


def compute_occupancy_grid_2d(
    points_for_occupancy: np.ndarray,
    cell_size: float = 0.06,
    bounds: Tuple[float, float, float, float] = (0.0, 1.0, 0.0, 1.0),
    cell_metadata_function: Optional[Callable[[np.ndarray, np.ndarray], Dict[str, Any]]] = None,
) -> OccupancyGrid2D:
    grid_info = compute_grid_info_2d(cell_size, bounds)
    filtered_points, cell_to_point_indices = group_points_by_cell_2d(points_for_occupancy, grid_info)

    if cell_metadata_function is None:
        cell_metadata_function = default_cell_metadata_function

    occupied_cells: List[Tuple[int, int]] = []
    cell_point_counts_list: List[int] = []
    cell_metadata_list: List[Dict[str, Any]] = []

    for (cell_index_x, cell_index_y), point_indices in cell_to_point_indices.items():
        cell_points = filtered_points[point_indices]
        cell_center = compute_cell_center_2d(cell_index_x, cell_index_y, grid_info)

        occupied_cells.append((cell_index_x, cell_index_y))
        cell_point_counts_list.append(int(len(point_indices)))
        cell_metadata_list.append(cell_metadata_function(cell_points, cell_center))

    cell_point_counts = np.array(cell_point_counts_list, dtype=int) if cell_point_counts_list else np.empty((0,), dtype=int)

    return OccupancyGrid2D(
        grid_info=grid_info,
        occupied_cells=occupied_cells,
        cell_point_counts=cell_point_counts,
        cell_metadata=cell_metadata_list,
    )


# =============================
# 4) Downsampling methods (independent of occupancy computation)
# =============================
def apply_random_sampling_2d(points: np.ndarray, num_points_to_keep: int = 500, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)

    if num_points_to_keep <= 0:
        return np.empty((0, 2))

    num_points_to_keep = min(num_points_to_keep, len(points))
    chosen_indices = rng.choice(len(points), size=num_points_to_keep, replace=False)
    return points[chosen_indices]


def apply_voxel_grid_sampling_2d(
    points: np.ndarray,
    cell_size: float = 0.06,
    bounds: Tuple[float, float, float, float] = (0.0, 1.0, 0.0, 1.0),
) -> np.ndarray:
    grid_info = compute_grid_info_2d(cell_size, bounds)
    filtered_points, cell_to_point_indices = group_points_by_cell_2d(points, grid_info)

    sampled_points_list: List[np.ndarray] = []
    for cell_index, point_indices in cell_to_point_indices.items():
        cell_points = filtered_points[point_indices]
        sampled_points_list.append(cell_points.mean(axis=0))

    return np.array(sampled_points_list) if sampled_points_list else np.empty((0, 2))


def apply_uniform_sampling_2d(
    points: np.ndarray,
    cell_size: float = 0.06,
    bounds: Tuple[float, float, float, float] = (0.0, 1.0, 0.0, 1.0),
) -> np.ndarray:
    grid_info = compute_grid_info_2d(cell_size, bounds)
    filtered_points, cell_to_point_indices = group_points_by_cell_2d(points, grid_info)

    sampled_points_list: List[np.ndarray] = []
    for (cell_index_x, cell_index_y), point_indices in cell_to_point_indices.items():
        cell_points = filtered_points[point_indices]
        cell_center = compute_cell_center_2d(cell_index_x, cell_index_y, grid_info)

        squared_distances = np.sum((cell_points - cell_center) ** 2, axis=1)
        sampled_points_list.append(cell_points[int(np.argmin(squared_distances))])

    return np.array(sampled_points_list) if sampled_points_list else np.empty((0, 2))


# =============================
# 5) Occupancy confidence (extensible)
# =============================
def create_occupancy_confidence_function(strategy: str = "binary", power: float = 1.0) -> Callable[..., float]:
    """
    confidence(point_count, max_point_count, **cell_metadata) -> [0,1]
    """
    def occupancy_confidence(point_count: int, max_point_count: int, **cell_metadata: Any) -> float:
        if max_point_count <= 0:
            return 0.0

        if strategy == "binary":
            return 1.0 if point_count > 0 else 0.0

        if strategy == "linear":
            return float(point_count) / float(max_point_count)

        if strategy == "log":
            return float(np.log1p(point_count) / np.log1p(max_point_count))

        if strategy == "power":
            normalized = float(point_count) / float(max_point_count)
            return float(normalized ** power)

        raise ValueError("Unknown strategy. Use: binary, linear, log, power")

    return occupancy_confidence


# =============================
# 6) Visualization
# =============================
def plot_scene_2d(
    axis,
    original_points: np.ndarray,
    sampled_points: np.ndarray,
    occupancy_grid: Optional[OccupancyGrid2D],
    title: str,
    occupancy_confidence_function: Optional[Callable[..., float]] = None,
    base_fill_alpha: float = 0.40,
    fill_colormap_name: str = "viridis",
    show_grid_lines: bool = True,
    original_point_size: int = 6,
    sampled_point_size: int = 26,
):
    axis.set_title(title)
    axis.set_aspect("equal", adjustable="box")

    # Occupancy visualization (if provided)
    if occupancy_grid is not None:
        grid_info = occupancy_grid.grid_info
        x_min, x_max, y_min, y_max = grid_info.bounds
        cell_size = grid_info.cell_size

        axis.set_xlim(x_min, x_max)
        axis.set_ylim(y_min, y_max)

        if show_grid_lines:
            for cell_index_x in range(grid_info.num_cells_x + 1):
                x_value = x_min + cell_index_x * cell_size
                axis.plot([x_value, x_value], [y_min, y_max], color="black", lw=0.4, alpha=0.15, zorder=1)
            for cell_index_y in range(grid_info.num_cells_y + 1):
                y_value = y_min + cell_index_y * cell_size
                axis.plot([x_min, x_max], [y_value, y_value], color="black", lw=0.4, alpha=0.15, zorder=1)

        if occupancy_confidence_function is None:
            occupancy_confidence_function = create_occupancy_confidence_function(strategy="binary")

        max_point_count = int(occupancy_grid.cell_point_counts.max()) if len(occupancy_grid.cell_point_counts) > 0 else 0
        colormap = plt.get_cmap(fill_colormap_name)

        for (cell_index_x, cell_index_y), point_count, metadata in zip(
            occupancy_grid.occupied_cells,
            occupancy_grid.cell_point_counts,
            occupancy_grid.cell_metadata,
        ):
            confidence = occupancy_confidence_function(
                point_count=int(point_count),
                max_point_count=max_point_count,
                **metadata
            )
            confidence = float(np.clip(confidence, 0.0, 1.0))

            face_color = colormap(confidence)
            fill_alpha = base_fill_alpha * confidence

            rectangle_x = x_min + cell_index_x * cell_size
            rectangle_y = y_min + cell_index_y * cell_size

            axis.add_patch(Rectangle(
                (rectangle_x, rectangle_y),
                cell_size, cell_size,
                facecolor=face_color,
                edgecolor="none",
                alpha=fill_alpha,
                zorder=0,
            ))
    else:
        # If no occupancy grid is provided, auto bounds
        x_min, y_min = original_points.min(axis=0)
        x_max, y_max = original_points.max(axis=0)
        padding = 0.03
        axis.set_xlim(x_min - padding, x_max + padding)
        axis.set_ylim(y_min - padding, y_max + padding)

    # Points
    axis.scatter(
        original_points[:, 0], original_points[:, 1],
        s=original_point_size, c="tab:blue", alpha=0.45,
        label="original", zorder=2
    )
    axis.scatter(
        sampled_points[:, 0], sampled_points[:, 1],
        s=sampled_point_size, c="crimson", alpha=0.95,
        label="sampled", zorder=3
    )
    axis.legend(loc="upper right", frameon=True)


def choose_occupancy_points_2d(
    original_points: np.ndarray,
    sampled_points: np.ndarray,
    source: str = "sampled",  # "sampled" | "original"
) -> np.ndarray:
    if source == "sampled":
        return sampled_points
    if source == "original":
        return original_points
    raise ValueError("source must be 'sampled' or 'original'")

    
# =============================
# 7) Demo: worst -> best, with flexible occupancy source
# =============================
if __name__ == "__main__":
    points = generate_point_cloud_2d(num_points=2500, seed=7)

    bounds = (0.0, 1.0, 0.0, 1.0)
    cell_size = 0.06

    # Downsampling (independent)
    random_sampled_points = apply_random_sampling_2d(points, num_points_to_keep=500, seed=123)
    voxel_grid_sampled_points = apply_voxel_grid_sampling_2d(points, cell_size=cell_size, bounds=bounds)
    uniform_sampled_points = apply_uniform_sampling_2d(points, cell_size=cell_size, bounds=bounds)

    # Per-panel: what drives occupancy visualization
    occupancy_source_random = "sampled"   # try also: "original"
    occupancy_source_voxel = "sampled"    # try also: "original"
    occupancy_source_uniform = "sampled"  # try also: "original"

    occupancy_grid_random = compute_occupancy_grid_2d(
        choose_occupancy_points_2d(points, random_sampled_points, source=occupancy_source_random),
        cell_size=cell_size,
        bounds=bounds,
    )
    occupancy_grid_voxel = compute_occupancy_grid_2d(
        choose_occupancy_points_2d(points, voxel_grid_sampled_points, source=occupancy_source_voxel),
        cell_size=cell_size,
        bounds=bounds,
    )
    occupancy_grid_uniform = compute_occupancy_grid_2d(
        choose_occupancy_points_2d(points, uniform_sampled_points, source=occupancy_source_uniform),
        cell_size=cell_size,
        bounds=bounds,
    )

    occupancy_confidence_linear = create_occupancy_confidence_function(strategy="linear")
    occupancy_confidence_binary = create_occupancy_confidence_function(strategy="binary")

    figure, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=300)

    plot_scene_2d(
        axis=axes[0],
        original_points=points,
        sampled_points=random_sampled_points,
        occupancy_grid=occupancy_grid_random,
        title=f"1) Random sampling (occupancy source: {occupancy_source_random})",
        occupancy_confidence_function=occupancy_confidence_binary,
        fill_colormap_name="Greys",
        base_fill_alpha=0.45,
        show_grid_lines=True,
    )

    plot_scene_2d(
        axis=axes[1],
        original_points=points,
        sampled_points=voxel_grid_sampled_points,
        occupancy_grid=occupancy_grid_voxel,
        title=f"2) Voxel grid mean (cell_size={cell_size})\noccupancy source: {occupancy_source_voxel}, intensity: linear",
        occupancy_confidence_function=occupancy_confidence_linear,
        fill_colormap_name="viridis",
        base_fill_alpha=0.45,
        show_grid_lines=True,
    )

    plot_scene_2d(
        axis=axes[2],
        original_points=points,
        sampled_points=uniform_sampled_points,
        occupancy_grid=occupancy_grid_uniform,
        title=f"3) Uniform (closest-to-center, cell_size={cell_size})\noccupancy source: {occupancy_source_uniform}, intensity: binary",
        occupancy_confidence_function=occupancy_confidence_binary,
        fill_colormap_name="plasma",
        base_fill_alpha=0.45,
        show_grid_lines=True,
    )

    plt.tight_layout()
    plt.show()

<a id='pcl_features_estimation'></a>
### Оценка признаков<sup>[toc](#_toc)</sup>

In [ ]:
import numpy as np

def _rng(seed=None):
    return np.random.default_rng(seed)

def _orthonormal_basis_from_normal(n):
    """Вернёт два ортонормированных вектора u,v в плоскости, перпендикулярной n."""
    n = np.asarray(n, dtype=float)
    n = n / (np.linalg.norm(n) + 1e-12)
    # выберем вектор, не параллельный n
    a = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    u = np.cross(n, a)
    u = u / (np.linalg.norm(u) + 1e-12)
    v = np.cross(n, u)
    v = v / (np.linalg.norm(v) + 1e-12)
    return u, v

def gen_plane(n=5000, center=(0,0,0), normal=(0,0,1), size=(2.0, 2.0), noise=0.01, seed=0):
    """Плоскость: точки равномерно по прямоугольнику в базисе (u,v)."""
    rng = _rng(seed)
    center = np.asarray(center, float)
    u, v = _orthonormal_basis_from_normal(normal)
    su, sv = size
    a = rng.uniform(-su/2, su/2, size=n)
    b = rng.uniform(-sv/2, sv/2, size=n)
    pts = center + a[:,None]*u[None,:] + b[:,None]*v[None,:]
    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

def gen_sphere(n=5000, center=(0,0,0), radius=1.0, noise=0.005, seed=0):
    """Сфера: точки равномерно по поверхности (по направлениям)."""
    rng = _rng(seed)
    center = np.asarray(center, float)
    dirs = rng.normal(size=(n,3))
    dirs /= (np.linalg.norm(dirs, axis=1, keepdims=True) + 1e-12)
    pts = center + radius * dirs
    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

def gen_cylinder(n=5000, center=(0,0,0), radius=1.0, height=2.0, axis=(0,0,1), noise=0.005, seed=0):
    """Цилиндр: боковая поверхность (без крышек). axis должен быть (примерно) единичным."""
    rng = _rng(seed)
    center = np.asarray(center, float)
    axis = np.asarray(axis, float)
    axis = axis / (np.linalg.norm(axis) + 1e-12)

    # базис, перпендикулярный оси
    u, v = _orthonormal_basis_from_normal(axis)

    theta = rng.uniform(0, 2*np.pi, size=n)
    t = rng.uniform(-height/2, height/2, size=n)

    circle = (np.cos(theta)[:,None]*u[None,:] + np.sin(theta)[:,None]*v[None,:]) * radius
    pts = center + circle + t[:,None]*axis[None,:]
    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

def gen_corner_two_planes(n=6000, size=2.0, noise=0.01, seed=0):
    """
    Две перпендикулярные плоскости: x=0 и y=0 (угол/ребро).
    Полезно для демонстрации проблем нормалей на ребре.
    """
    rng = _rng(seed)
    n1 = n // 2
    n2 = n - n1

    # Плоскость x=0, (y,z) в квадрате
    y1 = rng.uniform(-size/2, size/2, size=n1)
    z1 = rng.uniform(-size/2, size/2, size=n1)
    p1 = np.stack([np.zeros(n1), y1, z1], axis=1)

    # Плоскость y=0, (x,z) в квадрате
    x2 = rng.uniform(-size/2, size/2, size=n2)
    z2 = rng.uniform(-size/2, size/2, size=n2)
    p2 = np.stack([x2, np.zeros(n2), z2], axis=1)

    pts = np.vstack([p1, p2])
    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

def gen_sine_surface(n=5000, xlim=(-2,2), ylim=(-2,2), amp=0.3, fx=1.5, fy=1.0, noise=0.01, seed=0):
    """Гладкая “волнистая” поверхность z = amp*sin(fx*x) + amp*0.7*cos(fy*y)."""
    rng = _rng(seed)
    x = rng.uniform(xlim[0], xlim[1], size=n)
    y = rng.uniform(ylim[0], ylim[1], size=n)
    z = amp*np.sin(fx*x) + (amp*0.7)*np.cos(fy*y)
    pts = np.stack([x,y,z], axis=1)
    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

def gen_box_surfaces(n=8000, size=(2,2,2), noise=0.005, seed=0):
    """Поверхность прямоугольного параллелепипеда (6 граней)."""
    rng = _rng(seed)
    sx, sy, sz = size
    n_face = n // 6
    rem = n - 6*n_face

    def face_x(xconst, m):
        y = rng.uniform(-sy/2, sy/2, size=m)
        z = rng.uniform(-sz/2, sz/2, size=m)
        return np.stack([np.full(m, xconst), y, z], axis=1)

    def face_y(yconst, m):
        x = rng.uniform(-sx/2, sx/2, size=m)
        z = rng.uniform(-sz/2, sz/2, size=m)
        return np.stack([x, np.full(m, yconst), z], axis=1)

    def face_z(zconst, m):
        x = rng.uniform(-sx/2, sx/2, size=m)
        y = rng.uniform(-sy/2, sy/2, size=m)
        return np.stack([x, y, np.full(m, zconst)], axis=1)

    faces = [
        face_x(-sx/2, n_face), face_x(sx/2, n_face),
        face_y(-sy/2, n_face), face_y(sy/2, n_face),
        face_z(-sz/2, n_face), face_z(sz/2, n_face),
    ]
    pts = np.vstack(faces)

    if rem > 0:
        pts = np.vstack([pts, face_z(sz/2, rem)])

    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

### Дополнительные генераторы

#### 1. Плоскость + выбросы (равномерные 3D outliers)

In [ ]:
def gen_plane_with_outliers(n_in=6000, n_out=400,
                            plane_center=(0,0,0), plane_normal=(0,0,1), plane_size=(4,4),
                            outlier_box=((-2,-2,-2),(2,2,2)),
                            noise=0.01, seed=0):
    rng = np.random.default_rng(seed)

    # inliers: плоскость
    pts_in = gen_plane(n=n_in, center=plane_center, normal=plane_normal,
                       size=plane_size, noise=noise, seed=seed)

    # outliers: равномерно в параллелепипеде
    lo = np.asarray(outlier_box[0], float)
    hi = np.asarray(outlier_box[1], float)
    pts_out = rng.uniform(lo, hi, size=(n_out, 3))

    return np.vstack([pts_in, pts_out])

#### 2. Неравномерная плотность на одной поверхности (плоскость: слева плотнее, справа разреженнее)
Полезно, чтобы показать, что radius может “проваливаться” (мало соседей) в разреженной зоне, а kNN — нет.

In [ ]:
def gen_plane_two_density(n_dense=5000, n_sparse=800,
                          dense_region_x=(-2.0, 0.0),
                          sparse_region_x=(0.0, 2.0),
                          y_range=(-2.0, 2.0),
                          z0=0.0,
                          noise=0.01, seed=0):
    rng = np.random.default_rng(seed)

    # dense half-plane (x in dense_region_x)
    xd = rng.uniform(dense_region_x[0], dense_region_x[1], size=n_dense)
    yd = rng.uniform(y_range[0], y_range[1], size=n_dense)
    zd = np.full(n_dense, z0)

    # sparse half-plane (x in sparse_region_x)
    xs = rng.uniform(sparse_region_x[0], sparse_region_x[1], size=n_sparse)
    ys = rng.uniform(y_range[0], y_range[1], size=n_sparse)
    zs = np.full(n_sparse, z0)

    pts = np.vstack([
        np.stack([xd, yd, zd], axis=1),
        np.stack([xs, ys, zs], axis=1),
    ])
    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

#### 3. Плоскость с “отверстием” (отсутствие данных в зоне)
Показывает, как kNN может брать соседей “через отверстие” (далеко), а radius &mdash; не будет.

In [ ]:
def gen_plane_with_hole(n=7000,
                        xlim=(-2,2), ylim=(-2,2),
                        hole_center=(0.0, 0.0), hole_radius=0.6,
                        z0=0.0, noise=0.01, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(xlim[0], xlim[1], size=n*2)   # с запасом, потом отфильтруем
    y = rng.uniform(ylim[0], ylim[1], size=n*2)
    z = np.full_like(x, z0)

    pts = np.stack([x,y,z], axis=1)
    c = np.array([hole_center[0], hole_center[1]])
    mask = np.linalg.norm(pts[:, :2] - c[None, :], axis=1) > hole_radius
    pts = pts[mask][:n]
    pts += rng.normal(scale=noise, size=pts.shape)
    return pts

#### 4. “Inliers + clustered outliers” (выбросы не равномерно, а кластером)
Такой кейс хорошо демонстрирует, что локальная PCA может быть “утянута” кластером.

In [ ]:
def gen_plane_with_clustered_outliers(n_in=6000, n_out=600,
                                      plane_size=(4,4), noise=0.01,
                                      cluster_center=(1.0, 1.0, 0.8),
                                      cluster_sigma=0.08,
                                      seed=0):
    rng = np.random.default_rng(seed)
    pts_in = gen_plane(n=n_in, normal=(0,0,1), size=plane_size, noise=noise, seed=seed)

    cc = np.asarray(cluster_center, float)
    pts_out = cc[None, :] + rng.normal(scale=cluster_sigma, size=(n_out, 3))
    return np.vstack([pts_in, pts_out])

### Оценка нормалей

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

def estimate_normals(points: np.ndarray,
                     indices=None,
                     k: int | None = 30,
                     radius: float | None = None,
                     viewpoint=None):
    """
    points: (M,3)
    indices: None или массив индексов
    k: число соседей (используется, если radius is None)
    radius: радиус окрестности (если задан, k игнорируется)
    viewpoint: (3,) для согласования направления нормалей (опционально)

    return: (N,6) -> [x,y,z,nx,ny,nz] для запрошенных indices
    """
    points = np.asarray(points, dtype=float)
    assert points.ndim == 2 and points.shape[1] == 3

    M = points.shape[0]
    if indices is None:
        indices = np.arange(M)
    else:
        indices = np.asarray(indices, dtype=int)

    if (radius is None and k is None) or (radius is not None and k is not None):
        raise ValueError("Нужно задать ровно один параметр: либо k, либо radius (второй должен быть None).")

    tree = cKDTree(points)
    normals = np.full((len(indices), 3), np.nan, dtype=float)

    for i, idx in enumerate(indices):
        p = points[idx]

        if radius is not None:
            nbr_idx = tree.query_ball_point(p, r=float(radius))
            # гарантируем, что хотя бы сама точка есть
            if idx not in nbr_idx:
                nbr_idx.append(idx)
        else:
            kk = int(min(k, M))
            _, nbr_idx = tree.query(p, k=kk)
            nbr_idx = np.atleast_1d(nbr_idx).tolist()

        if len(nbr_idx) < 3:
            continue  # оставляем NaN

        nbrs = points[nbr_idx]
        c = nbrs.mean(axis=0)
        X = nbrs - c
        C = (X.T @ X) / max(len(nbrs) - 1, 1)

        w, V = np.linalg.eigh(C)
        n = V[:, np.argmin(w)]
        n = n / (np.linalg.norm(n) + 1e-12)

        if viewpoint is not None:
            vp = np.asarray(viewpoint, dtype=float)
            to_vp = vp - p
            if np.dot(n, to_vp) < 0:
                n = -n

        normals[i] = n

    return np.hstack([points[indices], normals])

In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_points_and_normals(cloud6, normal_scale=0.2, point_size=2, max_cones=1500, seed=0):
    cloud6 = np.asarray(cloud6, float)
    xyz = cloud6[:, 0:3]
    nrm = cloud6[:, 3:6]

    # уберём NaN-нормали
    ok = np.isfinite(nrm).all(axis=1)
    xyz_ok = xyz[ok]
    nrm_ok = nrm[ok]

    # подвыборка конусов для читаемости
    if len(xyz_ok) > max_cones:
        rng = np.random.default_rng(seed)
        sel = rng.choice(len(xyz_ok), size=max_cones, replace=False)
        xyz_ok = xyz_ok[sel]
        nrm_ok = nrm_ok[sel]

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=xyz[:,0], y=xyz[:,1], z=xyz[:,2],
        mode="markers",
        marker=dict(size=point_size, color="black"),
        name="points"
    ))
    fig.add_trace(go.Cone(
        x=xyz_ok[:,0], y=xyz_ok[:,1], z=xyz_ok[:,2],
        u=nrm_ok[:,0], v=nrm_ok[:,1], w=nrm_ok[:,2],
        anchor="tail",
        sizemode="scaled",
        sizeref=1.0/normal_scale,
        showscale=False,
        name="normals"
    ))
    fig.update_layout(scene=dict(aspectmode="data"), height=800)
    return fig

#### 1. Плоскость + нормали по kNN

In [ ]:
pts = gen_plane(n=6000, normal=(0,0,1), size=(4,4), noise=0.01, seed=1)
cloud6 = estimate_normals(pts, k=40, radius=None, viewpoint=(0,0,5))
plot_points_and_normals(cloud6, normal_scale=0.25).show()

#### 2. Сфера + нормали по радиусу

In [ ]:
pts = gen_sphere(n=8000, radius=1.0, noise=0.003, seed=2)
cloud6 = estimate_normals(pts, k=None, radius=0.15, viewpoint=(0,0,3))
plot_points_and_normals(cloud6, normal_scale=0.2).show()

#### 3. “Угол” из двух плоскостей (видно, что на ребре нормали неоднозначны)

In [ ]:
pts = gen_corner_two_planes(n=8000, size=3.0, noise=0.01, seed=3)
cloud6 = estimate_normals(pts, k=50, radius=None, viewpoint=(2,2,2))
plot_points_and_normals(cloud6, normal_scale=0.25).show()

<a id='geometrical_segmentation'></a>
### Геометрическая сегментация<sup>[toc](#_toc)</sup>

TODO

<a id='pose_estimation'></a>
# Оценка позы<sup>[toc](#_toc)</sup>
* [Point set registration](#point_set_registration)
* [Iterative Closest Point (ICP)](#icp)
* [Проблемы ICP](#icp_problems)

## Point set registration<sup>[toc](#_toc)</sup>

Как заматчить два облака?

<a id='icp'></a>
## Iterative Closest Point<sup>[toc](#_toc)</sup>

https://www.open3d.org/docs/release/tutorial/pipelines/icp_registration.html

__Идея__. Если хороших соответствий больше, чем плохих, то в процессе оптмизации их будет становиться все больше вплоть до сходимости процесса

__Алгоритм__
1. Хорошее начальное приближение. Возможно из constant velocity model или zero velocity motion model
2. Построение соответствий (association, correspondences estimation)
3. Оценка трансформа (transform estimation)
4. Продолжаем итерации вплоть до сходимости

__Варианты ICP__
* Point to point
* Point to plane
* Plane to plane
* Point to distr
* Distr to distr (GICP)

<a id='icp_problems'></a>
## Проблемы ICP<sup>[toc](#_toc)</sup>
* Выбросы и шумы в самих облаках
* Неточные соответствия
* Движения машины
* Движение вокруг машины

<a id='kitti'></a>
# KITTI<sup>[toc](#_toc)</sup>

Датасеты:
* https://www.thinkautonomous.ai/blog/lidar-datasets/
* https://www.argoverse.org/av2.html
* https://www.kaggle.com/datasets/klemenko/kitti-dataset/data

Мы будем работать с KITTI. Основной сайт https://www.cvlibs.net/datasets/kitti/

## Конфигурация<sup>[toc](#_toc)</sup>

![](https://www.cvlibs.net/datasets/kitti/images/setup_top_view.png)

![](https://www.cvlibs.net/datasets/kitti/images/passat_sensors_920.png)

<a id='kitti_loading'></a>
## Загрузка сцен<sup>[toc](#_toc)</sup>
Нужно скачать набор данных со страницы http://www.cvlibs.net/datasets/kitti/raw_data.php

Содержимое датасетов:
* Raw (unsynced+unrectified) and processed (synced+rectified) grayscale stereo sequences (0.5 Megapixels, stored in png format)
* Raw (unsynced+unrectified) and processed (synced+rectified) color stereo sequences (0.5 Megapixels, stored in png format)
* 3D Velodyne point clouds (100k points per frame, stored as binary float matrix)
* 3D GPS/IMU data (location, speed, acceleration, meta information, stored as text file)
* Calibration (Camera, Camera-to-GPS/IMU, Camera-to-Velodyne, stored as text file)
* 3D object tracklet labels (cars, trucks, trams, pedestrians, cyclists, stored as xml file)

#### 2011_09_26_drive_0002 (0.3 GB)
* [unsynced+unrectified data](https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0002/2011_09_26_drive_0002_extract.zip)
* [synced+rectified data](https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0002/2011_09_26_drive_0002_sync.zip)
* [calibration](https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_calib.zip)
* [tracklets](https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0002/2011_09_26_drive_0002_tracklets.zip)

#### 2011_09_26_drive_0106 (0.9 GB)
* [unsynced+unrectified data](https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0106/2011_09_26_drive_0106_extract.zip)
* [synced+rectified data](https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_drive_0106/2011_09_26_drive_0106_sync.zip)
* [calibration](https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_09_26_calib.zip)

Для удобной работы с выкаченным датасетом есть библиотека `pykitti` (https://github.com/utiasSTARS/pykitti)

In [ ]:
from sdc.kitti.dataset_adaptor import KittiDatasetAdaptor

In [ ]:
KITTI_DIR_PATH = os.path.abspath('./datasets/KITTI/')
print(f'KITTI dataset base dir: <{KITTI_DIR_PATH}>')

# Указываем датасет для загрузки
RIDE_DATE = '2011_09_26'
DRIVE = '0002'

# Загружаем данные. Опционально можно указать диапазон фреймов для загрузки
KITTI_DATASET = pykitti.raw(base_path=KITTI_DIR_PATH, date=RIDE_DATE, drive=DRIVE)

# Сразу же создаем адаптор
KITTI_DATASET_ADAPTOR = KittiDatasetAdaptor(KITTI_DATASET)

<a id='kitti_content'></a>
## Содержимое сцен<sup>[toc](#_toc)</sup>
* [Изображения](#kitti_images)
* [Облака](#kitti_clouds)
* [Локализация](#kitti_localization)
* [Калибровки](#kitti_calibrations)

Далее посмотрим на то, что содержится в загруженном датасете KITTI.

<a id='kitti_images'></a>
## Изображения<sup>[toc](#_toc)</sup>

**В датасете содержатся данные для 4-х камер**

In [ ]:
print(f'Number of camera 0 images: {len(KITTI_DATASET.cam0_files)}')
print(f'Number of camera 1 images: {len(KITTI_DATASET.cam1_files)}')
print(f'Number of camera 2 images: {len(KITTI_DATASET.cam2_files)}')
print(f'Number of camera 3 images: {len(KITTI_DATASET.cam3_files)}')

**Посмотрим на ректифицированные изображения камер:**

In [ ]:
frame_idx = 0
image0 = KITTI_DATASET.get_cam0(frame_idx)
image1 = KITTI_DATASET.get_cam1(frame_idx)
image2 = KITTI_DATASET.get_cam2(frame_idx)
image3 = KITTI_DATASET.get_cam3(frame_idx)

plt.figure(figsize=(20, 8))
plt.subplot(2, 2, 1)
plt.imshow(image0, cmap='gray')
plt.title('camera 0')

plt.subplot(2, 2, 2)
plt.imshow(image1, cmap='gray')
plt.title('camera 1')

plt.subplot(2, 2, 3)
plt.imshow(image2)
plt.title('camera 2')

plt.subplot(2, 2, 4)
plt.imshow(image3)
plt.title('camera 3');

plt.tight_layout()

del frame_idx, image0, image1, image2, image3

In [ ]:
frame_idx = 0
image = KITTI_DATASET.get_cam2(1)
print(f'Image type={type(image)}, image size={image.size}')
plt.figure(figsize=(16, 12))
plt.imshow(image)
del frame_idx, image

Ректифицированные изображения &mdash; это изображения в стерео-плоскости. Все камеры приведены к одной стерео-плоскости. Это нужно учитывать при проекции лидарных точек на ректифицированные изображения.

**Посмотрим на изображения цветной стерео-пары:**<sup>[toc](#_toc)</sup>

In [ ]:
frame_idx = 1
image1, image2 = KITTI_DATASET.get_rgb(frame_idx)
print(f'image1 type={type(image1)}, image1 size={image1.size}')
print(f'image2 type={type(image2)}, image2 size={image2.size}')

plt.figure(figsize=(16, 12))
plt.subplot(2, 1, 1)
plt.imshow(image1)
plt.subplot(2, 1, 2)
plt.imshow(image2);

<a id='kitti_clouds'></a>
### Облака<sup>[toc](#_toc)</sup>

In [ ]:
from sdc.pcl.tools.plotly_visualization import (
    create_plotly_figure,
    plot_cloud,
    apply_min_max_scaling,
    convert_values_to_rgba_tuples_f64,
)

# Загружаем облако из датасета
cloud_idx = 0
cloud_xyzi = KITTI_DATASET.get_velo(cloud_idx)
print(type(cloud_xyzi), cloud_xyzi.shape, cloud_xyzi.dtype)

# Визуализируем облако
figure = create_plotly_figure(bgcolor='black')

colors = apply_min_max_scaling(cloud_xyzi[:, 3], min_value=0.2, max_value=1.0)
colors = colors**(2/3.)  # Гамма-коррекция цветов 
colors = convert_values_to_rgba_tuples_f64(colors, cmap='hot')
assert colors.shape == (cloud_xyzi.shape[0], 4)

labels = [f'i={intensity:3f}' for intensity in cloud_xyzi[:, 3]]

plot_cloud(
    cloud=cloud_xyzi,
    colors=colors,
    figure=figure,
    labels=labels,
).show()

del colors, labels

#### Визуализация трех последовательных облаков<sup>[toc](#_toc)</sup>

In [ ]:
cloud0 = KITTI_DATASET.get_velo(0)
cloud1 = KITTI_DATASET.get_velo(1)
cloud2 = KITTI_DATASET.get_velo(2)

figure = create_plotly_figure(bgcolor='black')
plot_cloud(cloud0, colors='red', figure=figure)
plot_cloud(cloud1, colors='green', figure=figure)
plot_cloud(cloud2, colors='blue', figure=figure)
figure.show()

del cloud0, cloud1, cloud2

<a id='kitti_localization'></a>
### Локализация<sup>[toc](#_toc)</sup>

Результаты локализации лежат в `oxts` файлах. Описание формата можно посмотреть [здесь](https://github.com/pratikac/kitti/blob/master/readme.raw.txt).

Для преобразования между различными системами геопозиционирования нам потребуется библиотека `pyproj`, установка которой проведена во время настройки питоновского окружения.

Считывание показаний из датасета состоит из несольких шагов:
1. Считываем latitude, longitude, altitude положений GPS-сенсора (LLA-геопозиции)
2. Создаем [проекцию меркатора](https://en.wikipedia.org/wiki/Mercator_projection), где в качестве референсной точки выбираем первое показание GPS-сенсора
3. Конвертируем LLA-геопозиции в XYZ-геопозиции в системе координат меркатора
4. Считываем ориентации GPS-сенсора в системе координат меркатора
5. Формируем окончательные значения показаний локализации

Эти шаги детально рассматривались на лекции/семинаре. Здесь же просто воспользуемся классом `KittiDatasetAdaptor`, чтобы просто получить окончетельный набор поз GPS-сенсора.

In [ ]:
from sdc.geo.geo_lla_xyz_converter import GeoLlaXyzConverter
from sdc.geo.geo_position_lla import GeoPositionLLA
from sdc.geo.geo_position_xyz import GeoPositionXYZ
from sdc.kitti.dataset_adaptor import KittiDatasetAdaptor
from sdc.kitti.localization import (
    Localization,
    build_localization,
    build_localizations,
)
from sdc.kitti.convert_localization import (
    convert_localization_to_transform_matrix,
    convert_localizations_to_transform_matrices,
)

In [ ]:
KITTI_DATASET_ADAPTOR = KittiDatasetAdaptor(KITTI_DATASET)
reference_point = KITTI_DATASET_ADAPTOR.read_geo_positions_lla()[0]
localizations = KITTI_DATASET_ADAPTOR.build_localizations(reference_point)
# del reference_point

#### Пример работы GeoLlaXyzConverter-а

In [ ]:
# Инициализируем конвертер между LLA-геопозициями и XYZ-геопозициями в системе координат меркатора
geo_lla_xyz_converter = GeoLlaXyzConverter(reference_point)
print(f'geo_lla_xyz_converter.reference_point: {geo_lla_xyz_converter.reference_point}')

# Проверяем работу конвертера между проекцией меркатора и latlong-системой координат
print(geo_lla_xyz_converter.convert_xyz_to_lla(GeoPositionXYZ(0., 0., 0.)))
print(geo_lla_xyz_converter.convert_lla_to_xyz(geo_lla_xyz_converter.reference_point))

# Конвертируем LLA-геопозиции в XYZ-геопозиции
geo_positions_lla = KITTI_DATASET_ADAPTOR.read_geo_positions_lla()
geo_positions_xyz = [geo_lla_xyz_converter.convert_lla_to_xyz(lla) for lla in geo_positions_lla]

del geo_lla_xyz_converter

#### Визуализация траекторию, построенной по XYZ-позициями GPS-сенсора

In [ ]:
xs = [g.x for g in geo_positions_xyz]
ys = [g.y for g in geo_positions_xyz]
zs = [g.z for g in geo_positions_xyz]

fig = create_plotly_figure()
fig.add_scatter3d(**dict(
    x=xs,
    y=ys,
    z=zs,
    mode='lines+markers',
    line=dict(color='red', width=2),
    marker=dict(size=5),
))
fig.show()
del xs, ys, zs

<a id='kitti_calibrations'></a>
### Калибровки<sup>[toc](#_toc)</sup>
* Описание расположения сенсоров http://www.cvlibs.net/datasets/kitti/setup.php
* Описание содержимого файлов с калибровками https://github.com/yanii/kitti-pcl/blob/master/KITTI_README.TXT
    * `T_cam2_velo` &mdash; матрица перехода из лидара в камеру
    * `R_rect_20` &mdash; что это?
    * `P_rect_20` &mdash; матрица проекций ректифицированного изображения
  
 * Камера 0 выступает в качестве референсной системы координат. Положение лидара в системе координат камеры 0  содержится в файле `calib_velo_to_cam.txt` (аттрибут `T_cam0_velo_unrect`).


В KITTI в качестве ректифицированных изображений рассматриваются "выровненные" в плоскости стереопары изображения. Так как плоскость стереопары в общем случае повернута относительно исходной плоскости изображения камеры, то в дело вступает матрица ректификации. Матрица перехода из системы координат лидара в систему координат камеры $i$ имеет следующий вид:
$$
T_{l \to c_i} = R_{i} T_{c_0 \to c_i} T_{l \to c_0},
$$
где
* $T_{l \to c_0}$ &mdash; матрица перехода из системы координат лидара в систему координат камеры $0$
* $T_{c_0 \to c_i}$ &mdash; матрица перехода из системы координат лидара в систему координат камер $i$
* $R_i$ &mdash; матрица ректификации для камеры $i$ (матрица поворота, которая выравнивает камеры между собой, переводя их к одной плоскости изображения).

In [ ]:
obtained = np.dot(KITTI_DATASET.calib.R_rect_00, KITTI_DATASET.calib.T_cam0_velo_unrect) 
expected = KITTI_DATASET.calib.T_cam0_velo
assert np.max(np.abs(expected - obtained)) < 1e-10
print(KITTI_DATASET.calib.T_cam0_velo)

#### Матрицы ректификации

In [ ]:
for camera_idx in range(4):
    R_rect_name = 'R_rect_{}0'.format(camera_idx)
    print('{}:\n{}\n'.format(R_rect_name, getattr(KITTI_DATASET.calib, R_rect_name)))
    del camera_idx, R_rect_name

**Матрицы ректификации у всех камер отличаются**

#### Матрицы проекций

In [ ]:
P_rect = {}
for camera_idx in range(4):
    P_rect_name = 'P_rect_{}0'.format(camera_idx)
    P_rect[camera_idx] = getattr(KITTI_DATASET.calib, P_rect_name)
    print('{}:\n{}\n'.format(P_rect_name, P_rect[camera_idx]))
    del camera_idx

for first, second in itertools.product(range(4), range(4)):
    assert np.max(np.abs(P_rect[first] - P_rect[second])[:3, :3]) == 0.

* Основная часть у всех матриц проекций идентична, что, в принципе, логично, так как плоскость Z = 1 у стерео-камер идентична.
* Камера 2 единственная расположена левее камеры 0, именно поэтому элемент P[0, 3] у этой камеры положителен, в отличие от камер 1 и 3.

In [ ]:
camera_idx = 2
np.set_printoptions(precision=5)
L2C_name = f'T_cam{camera_idx}_velo'
K_name = f'K_cam{camera_idx}'
P_rect_name = f'P_rect_{camera_idx}0'
R_rect_name = f'R_rect_{camera_idx}0'

print('Lidar to camera transform:\n{}\n'.format(getattr(KITTI_DATASET.calib, L2C_name)))
print('{}:\n{}\n'.format(K_name, getattr(KITTI_DATASET.calib, K_name)))
print('{}:\n{}\n'.format(R_rect_name, getattr(KITTI_DATASET.calib, R_rect_name)))
print('{}:\n{}\n'.format(P_rect_name, getattr(KITTI_DATASET.calib, P_rect_name)))
del camera_idx

* https://medium.com/swlh/camera-lidar-projection-navigating-between-2d-and-3d-911c78167a94
* https://github.com/darylclimb/cvml_project/tree/master/projections/lidar_camera_projection

In [ ]:
from pykitti.utils import read_calib_file

CALIB = {}
CALIB['T_lidar_to_unrect0'] = KITTI_DATASET.calib.T_cam0_velo_unrect

cam_to_cam_calib = read_calib_file(os.path.join(KITTI_DIR_PATH, '2011_09_26', 'calib_cam_to_cam.txt'))
for camera_index in range(4):
    T = np.eye(4)
    T[:3, :3] = cam_to_cam_calib[f'R_0{camera_index}'].reshape(3, 3)
    T[:3, 3] = cam_to_cam_calib[f'T_0{camera_index}']
    CALIB[f'T_lidar_to_unrect{camera_index}'] = np.dot(T, CALIB['T_lidar_to_unrect0'])
    
    R_rect = np.eye(4)
    R_rect[:3, :3] = cam_to_cam_calib[f'R_rect_0{camera_index}'].reshape(3, 3)
    CALIB[f'T_lidar_to_rect{camera_index}'] = np.dot(R_rect, CALIB[f'T_lidar_to_unrect{camera_index}'])

In [ ]:
NUM_CAMERAS = 4
for camera_idx in range(NUM_CAMERAS):
    print(CALIB['T_lidar_to_rect1'])
    print(KITTI_DATASET.calib.T_cam1_velo)

#### Посмотрим на расположение камер

Камера 2 самая левая согласно расположению сенсоров.

In [ ]:
print(KITTI_DATASET.calib.T_cam2_velo - KITTI_DATASET.calib.T_cam0_velo)
print(KITTI_DATASET.calib.T_cam2_velo - KITTI_DATASET.calib.T_cam3_velo)
print(KITTI_DATASET.calib.T_cam2_velo - KITTI_DATASET.calib.T_cam1_velo)

Таким образом, все +- согласно ожидаемому расположению:
* Обе камеры стоят параллельно по осям
* Камера 2 стоит левее камеры 0 на 6.2 см
* Камера 2 стоит левее камеры 3 на 53.2 см

In [ ]:
print(KITTI_DATASET.calib.P_rect_20)

In [ ]:
np.set_printoptions(precision=5)
print(KITTI_DATASET.calib.T_cam2_velo)

In [ ]:
from sdc.project_points import (
    project_points_on_image,
    project_points_on_cylinder,
)

<a id='kitti_utils_projection_on_image'></a>
### Проектирование лидарных точек на изоображение<sup>[toc](#_toc)</sup>

In [ ]:
frame_idx = 34
camera_idx = 2

image = getattr(KITTI_DATASET, f'get_cam{camera_idx}')(frame_idx)
cloud = KITTI_DATASET.get_velo(frame_idx)
n_points = cloud.shape[0]

print('Image: type={}, size={}'.format(type(image), image.size))
print('Cloud: type={}, shape={}'.format(type(cloud), cloud.shape))

projected_points, depths, valid_points_mask = project_points_on_image(
    image=image,
    cloud=cloud,
    lidar_to_camera_transform=KITTI_DATASET.calib.T_cam0_velo,
    rectification_matrix=np.eye(4),
    projection_matrix=getattr(KITTI_DATASET.calib, f'P_rect_{camera_idx}0'),
)

print('Projected points shape: {}'.format(projected_points.shape))
print('Points in image canvas: {}'.format(np.sum(valid_points_mask)))

_, axes = plt.subplots(2, 1, figsize=(40, 20))

axes[0].imshow(image)
axes[1].imshow(image)
axes[1].scatter(
    projected_points[valid_points_mask, 0],
    projected_points[valid_points_mask, 1],
    c=depths[valid_points_mask],
    s=3)
del frame_idx, camera_idx

<a id='kitti_utils_projection_on_scan'></a>
### Projecting lidar points on scan<sup>[toc](#_toc)</sup>

In [ ]:
frame_idx = 34
image = KITTI_DATASET.get_cam2(frame_idx)
points_xyzi = KITTI_DATASET.get_velo(frame_idx)
print('Image is of type: {}'.format(type(image)))
print('Cloud is of type: {}'.format(type(points_xyzi)))

_, _, valid_points_mask = project_points_on_image(
    image=image,
    cloud=points_xyzi,
    lidar_to_camera_transform=KITTI_DATASET.calib.T_cam0_velo,
    rectification_matrix=np.eye(4),
    projection_matrix=KITTI_DATASET.calib.P_rect_20)
print('Points in image canvas: {}'.format(np.sum(valid_points_mask)))
print('Projected points shape: {}'.format(projected_points.shape))

valid_points_xyzi = points_xyzi[valid_points_mask]
valid_points_xyz = valid_points_xyzi[:, :3]
valid_depths = np.linalg.norm(valid_points_xyz, axis=1)
valid_intensities = valid_points_xyzi[:, 3]

num_lines = 64
num_columns = 640
scan, points_to_cylinder_map = project_points_on_cylinder(
    points=valid_points_xyzi,
    attributes=[0.05 * valid_depths, valid_intensities],
    num_lines=num_lines,
    num_columns=num_columns)
assert scan.shape == (3, num_lines, num_columns)

num_collisions = np.sum(scan[2] > 1)
num_points = np.sum(scan[2])
assert valid_points_xyz.shape[0] == num_points

print('points number:       {}'.format(points_xyzi.shape[0]))
print('valid points number: {}'.format(num_points))
print('collisions number:   {}'.format(num_collisions))
print('collisions rate:     {}'.format(float(num_collisions) / num_points))

plt.figure(figsize=(20, 10))
plt.imshow(np.sqrt(scan[0]), cmap='afmhot')

del frame_idx, image

<a id='sources'></a>
# Источники<sup>[toc](#_toc)</sup>
* [Блог Такси. Яндекс разрабатывает лидары](https://taxi.yandex.ru/blog/lidar)
* [Блог Яндекса. Беспилотный флот Яндекса перешёл на собственные лидары: почему это важно и что в них особенного](https://yandex.ru/blog/company/bespilotnyy-flot-yandeksa-pereshel-na-sobstvennye-lidary-pochemu-eto-vazhno-i-chto-v-nikh-osobennogo)

* [Хабр. Беспилотный автомобиль: оживляем алгоритмы. Доклад Яндекса](https://habr.com/ru/companies/yandex/articles/471636/)
* [Хабр. Как Яндекс делает обычные автомобили беспилотными](https://habr.com/ru/companies/yandex/articles/585444/)
* [Хабр. Нейронные сети для планирования движения беспилотных автомобилей](https://habr.com/ru/companies/yandex/articles/763348/)